#  Notebook 4 — SHAP Explainability

**Goal:** Load the saved Logistic Regression model and explain its predictions using SHAP.

**Input:** `Models/lr_model.pkl` + `Data/heart_cleaned.csv`

**SHAP Plots produced:**
1. Summary Plot (Beeswarm) — global feature importance
2. Bar Plot — mean absolute SHAP values
3. Waterfall Plot — single high-risk patient explained
4. Force Plot — individual prediction breakdown
5. Dependence Plot — how thalach interacts with ca

> Note: For Logistic Regression we use `shap.LinearExplainer`
> which is specifically designed for linear models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

# Load saved model and feature info
with open('../Models/lr_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../Models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../Models/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print(' Logistic Regression model, scaler and feature columns loaded')
print(f'   Feature count: {len(feature_cols)}')

## Step 1 — Prepare Data for SHAP

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Load cleaned dataset
df = pd.read_csv('../Data/heart_cleaned.csv')

# Encode — same as Notebook 3
ohe_cols = ['cp', 'restecg', 'slope', 'thal']
X = pd.get_dummies(df.drop(columns=['condition']), columns=ohe_cols, drop_first=True)
y = df['condition']

# Align columns to saved feature list
for col in feature_cols:
    if col not in X.columns:
        X[col] = 0
X = X[feature_cols]

# Same train/test split as Notebook 3
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale using the SAVED scaler
num_cols_to_scale = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak',
                     'thalach_pct_max', 'chol_age_ratio', 'oldpeak_slope']
scale_cols = [c for c in num_cols_to_scale if c in X_test.columns]

X_train_df = pd.DataFrame(X_train.values, columns=feature_cols)
X_test_df  = pd.DataFrame(X_test.values,  columns=feature_cols)

X_train_df[scale_cols] = scaler.transform(X_train_df[scale_cols])
X_test_df[scale_cols]  = scaler.transform(X_test_df[scale_cols])

# Convert to float
X_train_df = X_train_df.astype(float)
X_test_df  = X_test_df.astype(float)

print(f' Data ready for SHAP')
print(f'   Train: {X_train_df.shape}, Test: {X_test_df.shape}')

## Step 2 — Create SHAP Explainer

For Logistic Regression we use `shap.LinearExplainer`.
This is the correct explainer for linear models — it is faster and more accurate
than TreeExplainer for this model type.

In [ ]:
# LinearExplainer is designed for Logistic Regression and other linear models
explainer   = shap.LinearExplainer(model, X_train_df)
shap_values = explainer.shap_values(X_test_df)

print(' SHAP values computed using LinearExplainer')
print(f'   SHAP values shape: {shap_values.shape}')

## Step 3 — SHAP Summary Plot (Beeswarm)

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, plot_type='dot', show=False)
plt.title('SHAP Summary Plot — Global Feature Importance (Beeswarm)',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../App/static/plot_shap_summary.png', bbox_inches='tight')
plt.show()

print('Each dot = one patient.')
print('Horizontal position = how much that feature pushed the prediction.')
print('Colour = the actual value of that feature for that patient.')

## Step 4 — SHAP Bar Plot

In [ ]:
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Mean |SHAP Value|',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../App/static/plot_shap_bar.png', bbox_inches='tight')
plt.show()

## Step 5 — SHAP Waterfall Plot (Single High-Risk Patient)

In [ ]:
# Find the patient model is most confident is high risk
y_prob_test   = model.predict_proba(X_test_df)[:, 1]
high_risk_idx = int(np.argmax(y_prob_test))
actual_label  = list(y_test)[high_risk_idx]

print(f'Selected patient index : {high_risk_idx}')
print(f'Predicted probability  : {y_prob_test[high_risk_idx]:.2%}')
print(f'Actual label           : {"Disease" if actual_label == 1 else "No Disease"}')

shap_explanation = shap.Explanation(
    values        = shap_values[high_risk_idx],
    base_values   = explainer.expected_value,
    data          = X_test_df.iloc[high_risk_idx].values,
    feature_names = feature_cols
)

fig, ax = plt.subplots(figsize=(10, 6))
shap.plots.waterfall(shap_explanation, max_display=13, show=False)
plt.title(f'SHAP Waterfall — High-Risk Patient (prob: {y_prob_test[high_risk_idx]:.1%})',
          fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../App/static/plot_shap_waterfall.png', bbox_inches='tight')
plt.show()

## Step 6 — SHAP Force Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx],
    X_test_df.iloc[high_risk_idx],
    matplotlib=True,
    show=False,
    figsize=(14, 3)
)
plt.title('SHAP Force Plot — High-Risk Patient',
          fontweight='bold', fontsize=12, pad=40)
plt.tight_layout()
plt.savefig('../App/static/plot_shap_force.png', bbox_inches='tight')
plt.show()

## Step 7 — SHAP Dependence Plot

In [ ]:
plt.figure(figsize=(8, 5))
shap.dependence_plot(
    'thalach',
    shap_values,
    X_test_df,
    interaction_index='ca',
    show=False
)
plt.title('SHAP Dependence Plot: thalach (coloured by ca)',
          fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../App/static/plot_shap_dependence.png', bbox_inches='tight')
plt.show()

## Step 8 — Top Feature Importance Table

In [ ]:
mean_shap = pd.DataFrame({
    'Feature'    : feature_cols,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

mean_shap.index += 1
print('Top 15 Features by SHAP Importance:')
print(mean_shap.head(15).to_string())

mean_shap.to_csv('../Models/shap_importance.csv', index=False)
print('\n SHAP importance saved → Models/shap_importance.csv')

## Summary

In [ ]:
print('═'*55)
print('  SHAP ANALYSIS COMPLETE')
print('═'*55)
print('  Model explained: Logistic Regression')
print('  Explainer used: shap.LinearExplainer')
print()
print('  Saved plots:')
print('    App/static/plot_shap_summary.png')
print('    App/static/plot_shap_bar.png')
print('    App/static/plot_shap_waterfall.png')
print('    App/static/plot_shap_force.png')
print('    App/static/plot_shap_dependence.png')
print('    Models/shap_importance.csv')
print()
print('  Next step → python -m streamlit run App/app.py')
print('═'*55)